# VD20 — can the volatility engine forecast the next 24 hours, and does that beat implied?

Protocol: `discussions/Volatility_Dynamics/040_claude_vd20_next_24h_volatility_preregistration.md`.

Origin 08:00 UTC, target = realised variance over the next 24h (Σ r², 5m). Models: NAIVE (last 24h), ENGINE (VD18 HAR_4 = level of AM-T-HAR-Blend), HAR_D (1/5/22-day HAR), HAR_DX (HAR_D + intraday lags + leverage).
**G1** best of ENGINE/HAR_D beats NAIVE (QLIKE). **G2** ENGINE vs HAR_D. **G3** on VD19's 12:00 option rows: does the forecast add to implied?

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14, markov_har_variance as v18, model_vs_implied as mv, daily_vol_forecast as dv
for m in (v14,v18,mv,dv):importlib.reload(m)
SYNTHETIC=os.getenv('VD20_SYNTHETIC','0')=='1'
P=dv.Params(n_boot=400) if SYNTHETIC else dv.Params()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='vd20-');OUT=Path(tmp.name)/'20'
else:
    OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'20'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_daily_vol_forecast.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-400:]);assert chk.returncode==0,'VD20 tests failed'
print(json.dumps(dv.registration(P),indent=1,default=str))

## Engine refit (VD19 code, unchanged) and daily origins

In [ ]:
T0=time.time();H=sorted(set(dv.ORIGINS.values()))
if SYNTHETIC:
    raw_all=dv.synthetic_returns(n_days=520,assets=('BTC','ETH','SOL'),seed=11,start='2025-04-01')
    raw_fit=raw_all[raw_all.state_time<pd.Timestamp('2026-08-01',tz='UTC')];ASSETS=['BTC','ETH','SOL']
else:
    HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
    raw_fit,excl=v14.load_returns(HIST,mv.FIT_ASSETS,end=pd.Timestamp(mv.Params().fit_sample_end))
    raw_all,_=v14.load_returns(HIST,dv.ASSETS,end=pd.Timestamp(P.forecast_end),min_days=365);ASSETS=list(dv.ASSETS)
    print('excluded from engine fit:',excl)
frame=mv.build_frame(raw_fit,H,v14,v18)
FITS=mv.fit_har_models(frame,H,v18)
TRAIN_END=pd.Timestamp(frame.loc[frame.split=='train','date'].max())
print('engine HAR_4 fitted at',H,'| TRAIN ends',TRAIN_END.date(),'| fit universe',sorted(frame.underlying.unique()))
del frame,raw_fit
PANEL=mv.forecast_panel(raw_all,v14,v18)
Ds={}
for hr,h in dv.ORIGINS.items():
    D=dv.daily_origins(raw_all,hr,h,P)
    o=D[['underlying','state_time']].assign(h=h)
    D['F_ENGINE']=mv.model_sigma(PANEL,o,FITS,v18).to_numpy()**2
    Ds[hr]=D
D8=Ds[dv.PRIMARY_HOUR]
tr8=D8[D8.date+pd.Timedelta(days=1)<=TRAIN_END]
FD={hr:dv.fit_daily(Ds[hr][Ds[hr].date+pd.Timedelta(days=1)<=TRAIN_END]) for hr in Ds}
for hr in Ds:Ds[hr]=pd.concat([Ds[hr],dv.predict_daily(Ds[hr],FD[hr])],axis=1)
D8=Ds[dv.PRIMARY_HOUR]
print({m:{k:np.round(v,3) for k,v in zip(['const',*FD[8][m]['cols']],FD[8][m]['beta'])} for m in dv.DAILY})
print(f'{time.time()-T0:.0f}s')

## G1/G2 — forecast accuracy over the next 24h (EVAL = after the engine's TRAIN)

In [ ]:
EV=D8[D8.date>TRAIN_END].copy();TR=D8[D8.date+pd.Timedelta(days=1)<=TRAIN_END].copy()
SC,E=dv.score_table(EV,list(dv.MODELS),P);display(SC.round(4))
PAIRS=[('ENGINE','NAIVE'),('HAR_D','NAIVE'),('HAR_DX','NAIVE'),('ENGINE','HAR_D'),('HAR_DX','HAR_D')]
PD=dv.pair_diffs(E,PAIRS,P);display(PD.round(4))
print('Per asset QLIKE:');display(E.assign(**{m:dv.qlike(E.rv_next,E[f'F_{m}']) for m in dv.MODELS}).groupby('underlying')[list(dv.MODELS)].mean().round(4))
E['period']=np.where(E.date<pd.Timestamp('2026-09-19',tz='UTC'),'pre_2026-09-19','post (live collector era)')
print('By period:');display(E.assign(**{m:dv.qlike(E.rv_next,E[f'F_{m}']) for m in dv.MODELS}).groupby('period')[list(dv.MODELS)].agg(['mean','size']).round(4))

## Volatile-day classification and tails (what an OTM buyer needs)

In [ ]:
CL=dv.classification(E,list(dv.MODELS),P);display(CL.round(3))
TRs=pd.concat([TR],axis=1);TT=dv.tail_table(TRs,E,list(dv.MODELS),P)
display(TT.pivot_table(index='k',columns='model',values=['observed','gaussian','student_t_train']).round(4))
best=SC.sort_values('qlike').iloc[0].model
E['dec']=pd.qcut(np.log(E[f'F_{best}']/E.groupby('underlying').rv_next.transform('median')),10,labels=False)
print(f'Next-24h |move| by decile of {best} forecast (relative to asset median RV):')
display(E.groupby('dec').apply(lambda g:pd.Series({'n':len(g),'pred_sd_pct':100*np.sqrt(g[f'F_{best}']).mean(),
    'real_sd_pct':100*np.sqrt(g.rv_next).mean(),'abs_move_pct':100*g.R_next.abs().mean(),
    'P|z|>1.5':(g.R_next.abs()>1.5*np.sqrt(g[f'F_{best}'])).mean()})).round(3))

## G3 — versus implied: VD19 12:00 entries (1200 min to the 08:00 expiry)

In [ ]:
MODELS3=list(dv.MODELS)
if SYNTHETIC:
    d,sm,rv=mv.synthetic_comparison(n_sessions=25,seed=5,model_skill=0.6)
    R19=pd.DataFrame({'session':d.session,'underlying':d.underlying,'entry_hour':12,'sigma_imp':d.iv_straddle_mid*np.sqrt(d['T']),'rv_path':rv,'R_net':d.get('R_net',0.0)})
    D12=Ds[12].copy();ss=sorted(R19.session.unique());dd=sorted(D12.date.unique())[-len(ss):]
    mp=dict(zip(dd,[pd.Timestamp(s,tz='UTC')-pd.Timedelta(days=1) for s in ss]));D12['date']=D12.date.map(mp)
    D12=D12.dropna(subset=['date']);R19['underlying']=R19.underlying.map(dict(zip(sorted(R19.underlying.unique()),ASSETS)))
else:
    R19=pd.read_csv(ROOT/'notebooks'/'Volatility_Dynamics'/'19'/'19_rows.csv',dtype={'session':str});D12=Ds[12]
M3,INFO,ECON,IMP=dv.implied_check(R19,D12,MODELS3,P,mv)
print('rows',len(M3),'sessions',M3.session.nunique());print('implied alone:',{k:round(v,3) for k,v in IMP.items() if isinstance(v,float)})
display(INFO.round(3).T);display(ECON.round(3))
print('Mean log(F_model / implied variance) at 12:00:');display(M3[[f'x_{m}' for m in MODELS3]].describe().round(3))

## Decision and save

In [ ]:
DEC=dv.decide(PD,INFO);DEC['train_end']=str(TRAIN_END.date());DEC['eval_days']=int(E.date.nunique());DEC['best_qlike_model']=best
print(json.dumps(DEC,indent=1,default=str))
if not SYNTHETIC:
    SC.to_csv(OUT/'20_scores.csv',index=False);PD.to_csv(OUT/'20_pairs.csv',index=False);CL.to_csv(OUT/'20_classification.csv',index=False)
    TT.to_csv(OUT/'20_tails.csv',index=False);INFO.to_csv(OUT/'20_vs_implied.csv',index=False);ECON.to_csv(OUT/'20_economics.csv',index=False)
    E[['date','underlying','rv_next','R_next',*[f'F_{m}' for m in dv.MODELS]]].to_csv(OUT/'20_rows.csv',index=False,float_format='%.6g')
    (OUT/'20_decision.json').write_text(json.dumps(DEC,indent=1,default=str))
    (OUT/'20_manifest.json').write_text(json.dumps({'registration':dv.registration(P),'daily_fits':FD[8],'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')